# The website wizards 

# Our website: 
https://ri-ru.github.io/ADA-website/

Here, we look at our design process for the website, and focus on some key moments that shaped our journey :-)


# The design process Chapter 1: Inspiration 


Initial inspiration and sketches (pictures of all of the users otgether, like the pixel art integration)

The first part of the job was to figure out the look of our datastory. We did sketches and mockups in canva to see how it would look.

it did not look good at first!

As a large team effort, We looked at reference together. Here's some beautiful datastories we used for inspiration

https://pudding.cool/2021/04/music-bubble/
https://www.denizcemonduygu.com/
https://www.karim.news/

(here I would like three images side by side that show the three different websites, and a screenshot)


# Chapter 2: How the website works 


The website is built with SvelteKit (Svelte 5). Without boring you with how Svelte is better than other frameworks (hint: it is).... It essentially ships optimized vanilla JS at build time (unlike react, for example, that ships the framework code to the browser). Svelte does it during compile time. --> tiny bundles, very fast.

Hosting: We use GitHub Pages for static hosting (no jekyll!!). The build process outputs pure HTML/CSS/JS files to the build directory, which GitHub serves directly. No server & no database 

unsure if below is necessary (keeping for now)

In [ ]:

const config = {
	compilerOptions: {
		runes: true
	},
	preprocess,
	kit: {
		adapter: adapterStatic({
			pages: 'build',
			assets: 'build',
			fallback: 'index.html',
			precompress: false,
			strict: false
		}),
		paths: {
			base: dev ? '' : '/ADA-website'
		}
	}
};

export default config;

# Chapter 3: Snippets of Implementations

## The visual story

Here, explain the IDEA behidn the datastory. that Jimmy is a known youtuber that buys an essentially get rich quick scheme course. Instead, surprise: we get a high quality data analysis that Jimmy can use himself to inspire his journey. 

## First steps

Building the website (JAN IMAGE SPEAKING) 


- used a pudding.cool template (based on inspiration). + MENTION FEATURES IT HAS HERE 
- IT'S IN THE README

- Hosting turned out to be the first roadblock.

(describe why it sucked here to do the hosting) 





## Data Storage

Google sheets system here - the way it works describe. + say that it helped us all collaborate together: no need to push for minor changes. synced all at once.  Why It's Cool: You can edit ALL dialogue in Google Docs without touching code. 

💬 The Dialogue System + Google Docs Integration
The dialogue system is a two-part magic trick: content lives in Google Docs, gets synced to JSON, then parsed into chat bubbles with pixel art avatars.

Step 1: Google Docs → JSON

We use ArchieML (a structured text format) in Google Docs. Team members write dialogue like thi

[intro_chat_1]
[eddie] Oh... a customer?

[intro_chat_2]
[jimmy] I just spent 999$ on your course with my mom's credit card...

Running npm run gdoc fetches the Doc, parses ArchieML, and outputs copy.json:

{
  "intro_chat_1": "[eddie] Oh... a customer?",
  "intro_chat_2": "[jimmy] I just spent 999$ on my mom's credit card...",
  ...
}


In Index.svelte, we parse the [speaker] text format:


=====================
from readme (for inspiration)


Google Docs and Sheets
Create a Google Doc or Sheet
Click Share -> Advanced -> Change... -> Anyone with this link
In the address bar, grab the ID - eg. "...com/document/d/1IiA5a5iCjbjOYvZVgPcjGzMy5PyfCzpPF-LnQdCdFI0/edit"
paste in the ID above into google.config.js, and set the filepath to where you want the file saved
If you want to do a Google Sheet, be sure to include the gid value in the url as well
Running npm run gdoc at any point (even in new tab while server is running) will fetch the latest from all Docs and Sheets.



## The dialogue system 

The Challenge: You needed chat bubbles.  show the pixel art on the correct side, parse dialogue, and support html, and look good. 



In [6]:

const speakers = {
    jimmy: { avatar: "assets/pixel_art/jan.gif", name: "Jimmy" }
};

function getSpeaker(id) {
    return speakers[id.toLowerCase()] || { avatar: "assets/pixel_art/default.gif", name: id };
}

function parseChatMessage(str) {
    const match = str.match(/^\[(\w+)\]\s*(.*)$/);
    if (match) {
        const speakerKey = match[1].toLowerCase();
        const text = match[2];
        const side = speakerKey === 'jimmy' ? 'right' : 'left';
        return { speaker: match[1], text, side };
    }
    return { speaker: "Unknown", text: str, side: "left" };
}


##<-- dialogue rendering component -->

{#snippet dialogue(messages)}
    <div class="dialogue">
        {#each messages as msg, i}
            <div class="msg {msg.side}" style="--delay: {i * 0.1}s">
                {#if msg.side === "left"}
                    <!-- Expert on LEFT -->
                    <div class="avatar-col">
                        <img class="avatar-img" src={getSpeaker(msg.speaker).avatar} alt={msg.speaker} />
                        <span class="speaker-name">{getSpeaker(msg.speaker).name}</span>
                    </div>
                    <div class="bubble bubble-left">
                        <p>{@html msg.text}</p>
                    </div>
                {:else}
                    <!-- Jimmy on RIGHT -->
                    <div class="bubble bubble-right">
                        <p>{@html msg.text}</p>
                    </div>
                    <div class="avatar-col">
                        <img class="avatar-img" src={getSpeaker(msg.speaker).avatar} alt={msg.speaker} />
                        <span class="speaker-name">{getSpeaker(msg.speaker).name}</span>
                    </div>
                {/if}
            </div>
        {/each}
    </div>
{/snippet}

#using it: 

{@render dialogue(introChat)}

SyntaxError: invalid syntax (2917671061.py, line 1)

## Component - the design & pixel art 

https://www.pixilart.com/draw drawn by hand

issue: making it crisp. 

unified color system: necessary because all plots are different

We use Style Dictionary to maintain color consistency across CSS and JavaScript. Design tokens are defined once, then generated for both environments.

Source of Truth: design_system.css




In [ ]:
#making it crisp 

.avatar-img {
    width: 76px;
    height: 80px;
    min-width: 76px;
    min-height: 80px;
    flex-shrink: 0;
    image-rendering: pixelated;
    image-rendering: crisp-edges;
    -webkit-image-rendering: pixelated;
    transform: translateZ(0);
}

## Category Comparisons

Users pick "Category A" and "Category B" and Charts need to update synchronously, Colors need to stay consistent, Scales need to be comparable

The Trick: Svelte 5 runes ($state, $effect) make this reactive. Change a dropdown → charts instantly update.

MADE THE BELOW PRETTIER - ITS NOT QUITE THERE YET. 

In [ ]:
let selectedCategory = $state(null);
let vizIframe = $state(null);

// Compare categories
let compareCatA = $state(null);
let compareCatB = $state(null);
let compareIframe = $state(null);

function selectCategory(cat) {
    selectedCategory = cat;
    // Send message to iframe to update visualization
    if (vizIframe && vizIframe.contentWindow) {
        vizIframe.contentWindow.postMessage({
            type: "selectCategory",
            category: cat ? cat.name : null
        }, "*");
    }
}

function selectCompareA(cat) {
    compareCatA = cat;
    sendCompareMessage();
}

function selectCompareB(cat) {
    compareCatB = cat;
    sendCompareMessage();
}

SyntaxError: invalid syntax (301563404.py, line 1)

## How charts are rendered 

How Charts Get Data

imported svelte components

and just imported

two chart types - echarts & chart js





In [ ]:
import KalanChart0 from "./KalanChart0.svelte"
import KalanChart1 from "./KalanChart1.svelte"
import Compare_1 from "./Compare_1.svelte"
import VisualizationFocus from "./VisualizationFocus.svelte"
// etc.

<KalanChart0/>
<Compare_1 />
<VisualizationDumbbell />

## job distribution

JAN: 

VERO: 
